In [13]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from transformers import DistilBertTokenizer, DistilBertForSequenceClassification, Trainer, TrainingArguments
from datasets import Dataset
import evaluate
from src.config import DATA_DIR

In [3]:
df = pd.read_csv(DATA_DIR / 'raw' / 'chatbot' / 'question_classif.csv')

In [4]:
df

,question,label_text,label
0,What are the recommended prerequisites for the...,question_rag,0
1,Does the cybersecurity course cover intrusion ...,question_rag,0
2,How can I enroll in the Python course?,question_rag,0
3,What are the main basic concepts covered in th...,question_rag,0
4,Does the React course include practical projec...,question_rag,0
...,...,...,...
93,Ask the travel agent for information on visa r...,send_message,1
94,Send a note to the supervisor expressing inter...,send_message,1
95,Write to the community leader suggesting impro...,send_message,1
96,Text the roommate to coordinate grocery shoppi...,send_message,1


In [5]:
questions = df['question']
label_text = df['label_text']
label = df['label']

In [6]:
train_text, test_text, train_label, test_label = train_test_split(questions, label, test_size=0.2)

In [7]:
train_dataset = Dataset.from_dict({'text': train_text, 'label': train_label})
test_dataset = Dataset.from_dict({'text': test_text, 'label': test_label})

In [8]:
tokenizer = DistilBertTokenizer.from_pretrained('distilbert-base-uncased')
model = DistilBertForSequenceClassification.from_pretrained('distilbert-base-uncased')

tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

config.json:   0%|          | 0.00/483 [00:00<?, ?B/s]

/home/arys/miniconda3/envs/dev_ia/lib/python3.10/site-packages/transformers/tokenization_utils_base.py:1601: FutureWarning: `clean_up_tokenization_spaces` was not set. It will be set to `True` by default. This behavior will be depracted in transformers v4.45, and will be then set to `False` by default. For more details check this issue: https://github.com/huggingface/transformers/issues/31884
  warnings.warn(


model.safetensors:   0%|          | 0.00/268M [00:00<?, ?B/s]

Some weights of DistilBertForSequenceClassification were not initialized from the model checkpoint at distilbert-base-uncased and are newly initialized: ['classifier.bias', 'classifier.weight', 'pre_classifier.bias', 'pre_classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


In [16]:
for param in model.base_model.parameters():
    param.requires_grad = False

In [25]:
def tokenize_function(examples):
    return tokenizer(examples['text'], padding="max_length", truncation=True)

In [26]:
train_dataset = train_dataset.map(tokenize_function)
test_dataset = test_dataset.map(tokenize_function)

Map:   0%|          | 0/78 [00:00<?, ? examples/s]

Map:   0%|          | 0/20 [00:00<?, ? examples/s]

In [27]:
training_args = TrainingArguments(output_dir='./output_chatbot')

In [28]:
metric = evaluate.load("accuracy")
def compute_metrics(eval_pred):
    logits, labels = eval_pred
    predictions = np.argmax(logits, axis=-1)
    return metric.compute(predictions=predictions, references=labels)

In [29]:
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=test_dataset,
    compute_metrics=compute_metrics
)

In [30]:
trainer.train()

Step,Training Loss


TrainOutput(global_step=30, training_loss=0.6172698974609375, metrics={'train_runtime': 15.6674, 'train_samples_per_second': 14.935, 'train_steps_per_second': 1.915, 'total_flos': 30997371285504.0, 'train_loss': 0.6172698974609375, 'epoch': 3.0})

In [32]:
# from huggingface_hub import login
# login()
# model.push_to_hub("Arys02/fine_tuned_tp8")

model.safetensors:   0%|          | 0.00/268M [00:00<?, ?B/s]

CommitInfo(commit_url='https://huggingface.co/Arys02/fine_tuned_tp8/commit/1f8c9d65730fbe630037b069634bbf32c6066f7f', commit_message='Upload DistilBertForSequenceClassification', commit_description='', oid='1f8c9d65730fbe630037b069634bbf32c6066f7f', pr_url=None, repo_url=RepoUrl('https://huggingface.co/Arys02/fine_tuned_tp8', endpoint='https://huggingface.co', repo_type='model', repo_id='Arys02/fine_tuned_tp8'), pr_revision=None, pr_num=None)

In [33]:
trainer.evaluate()

{'eval_loss': 0.5899554491043091,
 'eval_accuracy': 1.0,
 'eval_runtime': 1.44,
 'eval_samples_per_second': 13.889,
 'eval_steps_per_second': 2.083,
 'epoch': 3.0}